In [20]:
!pip install -q -U llama-index

In [21]:
!pip install -q -U llama-index-llms-ollama

In [22]:
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama

### Zero-Prompting 

here the data is given by the user and the computer has to answer the question given by the user to from the information provided by the user only 

### Few-Shot Prompting

same as **Zero-Prompting** but few examples are given to the model so that it can understand the better way of answering 

In [36]:
def zero_shot_prompt(context,question):
    llm = Ollama(
        model = "gemma3:4b",
        temperature = 0,
    )
    template_str = (
            "You are a expert AI agent assistant.\n"
            "Use ONLY the user provided content to anser the user's questions\n"
            "If the content is insufficient or does not mention the answer relay exactly:"
            "'Not enough information.'\n\n"
            "content:\n{context_str}\n\n"
            "User question:\n{question_str}\n\n"
            "Answering rules:\n"
            "1) Be consine and it should be of 3-4 lines only.\n"
            "2) Use bullet points for list.\n"
            "Final Answer: "
        )
    template = PromptTemplate(template_str)
    filled_prompt = template.format(
        context_str=context,
        question_str=question
    )
    output = llm.complete(prompt=filled_prompt)
    response = output.text
    return response
 
def few_shot_prompt(context,examples,question):
    llm = Ollama(
        model = "gemma3:4b",
        temperature = 0,
    )
    examples_str = "\n".join(
        f"Examples{i+1}:\ncontext:\n{ex.get("context","")}\n"
        f"Question:{ex.get("question","")}\n"
        f"Answer:{ex.get("answer","")}\n"
        for i,ex in enumerate(examples)
    )
    template_str = (
            "You are a expert AI agent assistant.\n"
            "Use ONLY the user provided content to anser the user's questions\n"
            "If the content is insufficient or does not mention the answer relay exactly:"
            "'Not enough information.'\n\n"
            "Follow this style of answering from the examples"
            "examples:\n{examples_str}\n"
            "---END OF EXAMPLES------\n"
            "content:\n{context_str}\n\n"
            "User question:\n{question_str}\n\n"
            "Answering rules:\n"
            "1) Be consine and it should be of 3-4 lines only.\n"
            "2) Use bullet points for list.\n"
            "Final Answer: "
    )
    prompt = PromptTemplate(template_str).format(
        examples_str = examples,
        context_str=context,
        question_str=question
    )
    output = llm.complete(prompt=prompt)
    response = output.text
    return response


In [37]:
#testing the model 

context = "Large Language Models (LLMs) are a type of artificial intelligence designed to work with and understand human language. They are based on deep learning techniques, especially the transformer architecture, which enables them to effectively process sequences of words or tokens.LLMs are trained on extremely large datasets that consist of text from books, articles, websites, and other digital content. This training process involves learning to predict the next word in a sentence, which helps the model develop a complex understanding of grammar, context, and semantic nuances.The architecture of LLMs typically consists of multiple layers of transformers, which use self-attention mechanisms to weigh the importance of each word relative to others in a sentence. This allows the model to capture dependencies and context over long text spans, making it very proficient at generating coherent and contextually relevant text.LLMs can be fine-tuned for specific tasks such as translation, summarization, question answering, text completion, and conversational agents. However, training LLMs requires substantial computational power due to the large number of parameters involved, which can be in the billions.Despite their powerful capabilities, LLMs also have limitations, including tendencies to produce plausible-sounding but incorrect outputs, sensitivity to input phrasing, and potential biases inherited from training data. Ethical considerations are critical when deploying LLMs in real-world applications, especially in sensitive areas.In essence, LLMs represent a significant advancement in natural language processing by enabling machines to interact with human language in a way that feels intuitive and natural."

question = "Explain what a Large Language Model is in simple terms."

examples = [
    {"question":"What is a blockchain?","answer":"A blockchain is like a digital ledger that records transactions across many computers in a way that is secure and hard to change."},
    {"question":"Explain what the internet is to a beginner.","answer":"The internet is a network that connects computers all around the world, allowing people to share information and communicate easily."},
    {"question":"What is artificial intelligence?","answer":" Artificial intelligence is when machines or computers are designed to think and learn like humans, so they can do tasks such as recognizing images or understanding speech"},
]


print("Zero-shot:", zero_shot_prompt(context,question))
print("Few-shot:", few_shot_prompt(context,examples,question))

Zero-shot: Large Language Models (LLMs) are AI systems designed to understand and work with human language.
*   They are built using deep learning and a special architecture called the transformer.
*   LLMs learn by predicting the next word in a sentence, gaining a strong grasp of language.
*   These models are used for tasks like translation and conversation.
Few-shot: [{'question': 'Explain what a Large Language Model is in simple terms.', 'answer': 'Large Language Models (LLMs) are a type of artificial intelligence that focuses on understanding and working with human language.\n*   They are trained on massive amounts of text data to predict the next word in a sentence.\n*   This allows them to generate coherent and contextually relevant text for tasks like translation and question answering.\n*   Despite their abilities, LLMs can sometimes produce incorrect outputs and may have biases from their training data.'}]

